In [1]:
from biosteam import main_flowsheet as F
import biosteam as bst
import thermosteam as tmo
import pandas as pd
import numpy as np


from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.systems.rcf_oil_purification import create_rcf_oil_purification_system
from lignin_saf.systems.monomer_purification import create_monomer_purification_system
from lignin_saf.systems.hdo import create_hdo_system
from lignin_saf.systems.cellulosic_ethanol_no_preatreatment import create_cellulosic_ethanol_system
from atj_saf.atj_bst.etj_no_facilities import create_etj_system_no_facilities
from lignin_saf.cellulosic_tea import create_cellulosic_ethanol_tea

from atj_saf.atj_bst.etj_settings import price_data





chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 541.7   # 2016 USD basis

# Poplar group must be defined before creating any stream that references it
chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])

# ── Area 200: RCF process ──────────────────────────────────────────────────
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()

# ── Area 300: Purification ─────────────────────────────────────────────────
rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_CRUDE_OUT)
monomer_purification_sys = create_monomer_purification_system(ins=F.PURE_OIL_OUT)
rcf_oil_purification_sys.simulate()
monomer_purification_sys.simulate()

# ── Area 400: Hydrodeoxygenation ───────────────────────────────────────────
hdo_system = create_hdo_system(ins=F.MON_MONOMERS_OUT)
hdo_system.simulate()

etoh_system = create_cellulosic_ethanol_system(ins=F.Carbohydrate_Pulp, add_denaturant=False)
etoh_system.simulate()

# No pretreatment_wastewater — only S401 stillage filtrate goes to WWT.
etoh_ww     = [F.unit.S401.outs[1]]
etoh_solids = [F.unit.S401.outs[0]]

# Removing the NH3 fraction of the ethanol output - in the future CBP will remove this anyways, so I've just modelled it as a splitter
nh3_splitter = bst.units.Splitter(ins = F.T703.outs[0], split = {'NH3':1.0} )
nh3_splitter.simulate()

# Ethanol to Jet system
etj_system = create_etj_system_no_facilities(ins = nh3_splitter.outs[1])
etj_system.simulate()


WWT = bst.create_conventional_wastewater_treatment_system('WWT', ins=[F.WW_10, F.WastePulp, F.RCF_WW_OUTS, F.WW_11, F.WW_12, F.HDO_WW, F.HDO_wash_water, F.ETJ_WW_OUTS] + etoh_ww)

for unit in WWT.units:
    if hasattr(unit, 'strict_moisture_content'):
        unit.strict_moisture_content = False

F.unit.PWC.ins[0] = WWT.outs[2]

solids_to_BT = bst.Mixer('MIX_BT_solids', ins=[WWT.outs[1]] + etoh_solids)


BT = bst.facilities.BoilerTurbogenerator('BT', fuel_price=prices['CH4'])


gas_mixer= bst.Mixer('MIX_BT_gas', ins=(WWT.outs[0], F.RCF_PSAWASTE_OUTS, F.HDO_purge_gases, F.ETJ_PSAWASTE_OUTS))

BT.ins[0] = solids_to_BT.outs[0]  # Connecting sludge to BT solids feed
BT.ins[1] = gas_mixer.outs[0]   # Connecting biogas from WW treatment and PSA waste gases from RCF

combined_saf = bst.units.Mixer(ins = (F.ETJ_SAF_OUT, F.HDO_CYCLOALKANES_OUT), outs = 'TOTAL_SAF', rigorous = True)


rcf_pure_mon_hdo_etoh_etj_system = bst.System(
    'RCF+HDO+Cellulosic_ETJ',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys, hdo_system, etoh_system, etj_system, combined_saf, WWT),
    facilities=[solids_to_BT, gas_mixer, BT],
)

rcf_pure_mon_hdo_etoh_etj_system.simulate()



c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: RCF_PUMP1> no pump type available at current power (2.34e+03 hp), head (3.2e+03 ft), kinematic viscosity (5.83e-07 m2/s), and NPSH (4.18 ft); assuming centrigugal pump
  warn(f'{repr(self)} no pump type available at current power '
c:\users\hwadg\onedrive - the pennsylvania state university\shi_wadgama_shared\models\atjspk\lignin_saf\ligsaf_units.py:411: CostWarning: <SolvolysisReactor: RCF_RXR1> Vertical vessel length (58.75 ft) is out of bounds (12 to 40 ft) for cost correlation
  self._vertical_vessel_design(
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\_unit.py:1241: CostWarning: <IsentropicCompressor: RCF_COMP1> power (1.48

In [2]:
F.Hydrogen_In.price = price_data['hydrogen']   # 8.46 USD/kg
F.RN.price = price_data['renewable_naphtha']   # 0.71 USD/kg
F.RD.price = price_data['renewable_diesel']    # 1.888 USD/kg

In [3]:
integrated_tea = create_cellulosic_ethanol_tea(rcf_pure_mon_hdo_etoh_etj_system)
mjsp = round(((integrated_tea.solve_price(F.TOTAL_SAF)*F.TOTAL_SAF.rho)/264.172),2)

print(f'The MSP for SAF range cycloalkanes is  {mjsp} USD/gal')

The MSP for SAF range cycloalkanes is  21.71 USD/gal


In [13]:
F.Hydrogen_In

Stream: Hydrogen_In to <HydrogenStorageTank: T102>
phase: 'g', T: 298.15 K, P: 3e+06 Pa
flow (kmol/hr): Hydrogen  164


In [10]:
F.RCF_H2_IN

Stream: RCF_H2_IN to <Mixer: RCF_MIX2>
phase: 'g', T: 353.15 K, P: 3e+06 Pa
flow (kmol/hr): Hydrogen  771


In [8]:
F.HDO_H2_IN

Stream: HDO_H2_IN to <Mixer: HDO_MIX1>
phase: 'g', T: 298.15 K, P: 3e+06 Pa
flow (kmol/hr): Hydrogen  239


In [ ]:
# Operating costs plot
methanol_price = F.Meoh_in.F_mass * prices['Methanol'] * integrated_tea.operating_hours
hydrogen_price = F.Hydrogen_In.F_mass * prices['Hydrogen'] * integrated_tea.operating_hours
poplar_price = F.Poplar_In.F_mass * prices['Feedstock'] * integrated_tea.operating_hours
ethyl_acetate_price = F.EthylAcetate_in.F_mass * prices['EthylAcetate'] * integrated_tea.operating_hours
hexane_price = F.Hexane_In.F_mass * prices['Hexane'] * integrated_tea.operating_hours
catalyst = F.RCF_Catalyst.F_mass * prices['NiC_catalyst'] * integrated_tea.operating_hours
#natural_gas = 5.35e+04 * 0.2612 * integrated_tea.operating_hours
material_arrz = np.array([methanol_price, hydrogen_price, poplar_price, 
                            ethyl_acetate_price, hexane_price, catalyst])

In [7]:
rcf_pure_mon_hdo_etoh_etj_system.heat_utilities

[<ethylene: -2.53e+06 kJ/hr, 189 kmol/hr, 84.1 USD/hr>,
 <high_pressure_steam: 0 kJ/hr, 0 kmol/hr, 0 USD/hr>,
 <natural_gas: 0 kJ/hr, 0 kmol/hr, 0 USD/hr>,
 <low_pressure_steam: -1.04e-07 kJ/hr, 2.27e-13 kmol/hr, -3.13e-13 USD/hr>,
 <cooling_water: -2.05e+09 kJ/hr, 1.4e+06 kmol/hr, 682 USD/hr>,
 <medium_pressure_steam: -4.66e-10 kJ/hr, 0 kmol/hr, -7.11e-15 USD/hr>,
 <propane: -4.61e+07 kJ/hr, 2.51e+03 kmol/hr, 608 USD/hr>,
 <chilled_brine: -5.49e+06 kJ/hr, 3.57e+03 kmol/hr, 44.7 USD/hr>,
 <chilled_water: -1.89e+08 kJ/hr, 1.25e+05 kmol/hr, 943 USD/hr>]

In [6]:
F.BT.results()

Boiler turbogenerator                                      Units        BT
Electricity           Power                                   kW -3.15e+04
                      Cost                                USD/hr -2.46e+03
Medium pressure steam Duty                                 kJ/hr -1.11e+07
                      Flow                               kmol/hr      -306
                      Cost                                USD/hr     -84.3
High pressure steam   Duty                                 kJ/hr -1.41e+09
                      Flow                               kmol/hr -4.39e+04
                      Cost                                USD/hr -1.39e+04
Natural gas           Duty                                 kJ/hr    -2e+08
                      Flow                               kmol/hr      -295
                      Cost                                USD/hr -1.03e+03
Low pressure steam    Duty                                 kJ/hr -1.17e+09
                      Flow                               kmol/hr -3.02e+04
                      Cost                                USD/hr -7.18e+03
Cooling water         Duty                                 kJ/hr  -2.5e+07
                      Flow                               kmol/hr  1.71e+04
                      Cost                                USD/hr      8.33
Fuel (inlet)          Flow                                 kg/hr  4.62e+04
                      Cost                                USD/hr  1.22e+04
Ash disposal (outlet) Flow                                 kg/hr  1.44e+03
                      Cost                                USD/hr      45.6
Design                Work                                    kW  3.93e+04
                      Flow rate                            kg/hr  1.37e+06
                      Ash disposal                         kg/hr  1.44e+03
Purchase cost         Baghouse bags                          USD  1.51e+05
                      Boiler                                 USD     8e+07
                      Deaerator                              USD   8.6e+05
                      Amine addition pkg                     USD  1.13e+05
                      Hot process water softener system      USD   2.2e+05
                      Turbogenerator                         USD  8.95e+06
Total purchase cost                                          USD  9.03e+07
Utility cost                                              USD/hr -1.24e+04